# HelioGuard Phase 6: SFT warm start (Colab T4)

Run the cells in order. The cell marked **Resume** is the same training command: run it again after a disconnect and training continues from the last checkpoint on Drive.

Setup once: Colab secret `GITHUB_TOKEN` (only needed if the repo is private) and a GPU runtime (T4). Nothing secret is stored in this notebook.

In [ ]:
# 1. Clone the repo (token from the Colab secret, never printed or saved in the remote URL)
import os, subprocess
from google.colab import userdata
REPO = "https://github.com/nithinraj49/helioguard"
if not os.path.exists("/content/helioguard"):
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None   # public repo: clone without a token
    url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
    subprocess.run(["git", "clone", "-q", url, "/content/helioguard"], check=True)
    subprocess.run(["git", "-C", "/content/helioguard", "remote", "set-url", "origin", REPO + ".git"], check=True)
%cd /content/helioguard
!git log --oneline -1

In [ ]:
# 2. Install (Unsloth first, then the project)
!pip -q install unsloth
!pip -q install -e ".[sft]" ortools

In [ ]:
# 3. Mount Drive: checkpoints and the final adapter live here, so they survive disconnects
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints"
OUT = "/content/drive/MyDrive/helioguard/sft_1p5b"
os.makedirs(CKPT, exist_ok=True)

In [ ]:
# 4. Regenerate the SFT traces (deterministic, CPU, train tasks only) and show the data card
!python scripts/generate_sft_traces.py --workers 2 | head -60

In [ ]:
# 5. Train (Resume: run this same cell again after a disconnect)
!python scripts/train_sft.py configs/sft_1p5b.json --checkpoint-dir "$CKPT" --out-dir "$OUT"

In [ ]:
# 6. Evaluate the adapter on the 50 held-out tasks (greedy + temperature 0.7 x 3 seeds, bootstrap CIs, gates)
!python -m helioguard.sft_eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter "$OUT/adapter" --out "$OUT/eval" --tag sft_1p5b

Copy `sft_1p5b_eval_summary.json` and `train_summary.json` from Drive back into `results/phase6/` in the repo to commit them. Do not commit checkpoints or adapters unless asked.